# Notebook 04 — Prediction Simulations and Scenario Analysis (What-If)
**Level:** 4 (optional differentiator)  
**Depends on:** Notebooks 01, 02, and 03 must have been run.  

---

## ⚠️ Simulation Disclaimer

> **This notebook performs PREDICTIVE SIMULATIONS, not causal analysis.**  
> All scenario predictions are outputs of a machine learning model trained on
> historical data. The model has learned statistical associations between
> operational variables and % Silica Concentrate — it does NOT model the
> physical or chemical mechanisms of the flotation process.
>
> Scenario results should be interpreted as: *"If a similar historical pattern
> had occurred with this input combination, the model would have predicted X."*
> They are NOT engineering recommendations or process control instructions.
> Any operational change must be validated by process engineers and
> metallurgists before implementation.

---

## Objectives
1. Select a representative reference observation from the test set.
2. Define multiple what-if scenarios (variable modifications).
3. Predict % Silica Concentrate for each scenario.
4. Compare and visualise results.
5. Discuss trade-offs, constraints, and risks.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.config import CFG
from src.train import load_model
from src.predict import (
    predict_single,
    build_scenario,
    apply_pct_change,
    compare_scenarios,
    sensitivity_sweep,
    save_scenario_table,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
FIGURES_PATH = Path(CFG['paths']['reports_figures'])
TARGET = CFG['target']

---
## 1. Load Model and Test Data

In [ ]:
model = load_model(folder='selected', cfg=CFG)

processed_path = Path(CFG['paths']['data_processed'])
X_test = pd.read_parquet(processed_path / 'X_test.parquet')
y_test = pd.read_parquet(processed_path / 'y_test.parquet')[TARGET]

print(f'Model: {type(model).__name__}')
print(f'Test set: {X_test.shape}')

---
## 2. Select Reference Observation

We select a median-quality observation from the test set as our reference point.
This avoids edge cases (extreme high/low) and represents a typical operational state.

In [ ]:
# Select median-predicted observation from the test set
y_pred_test = model.predict(X_test)
pred_series = pd.Series(y_pred_test, index=X_test.index)

# Row closest to the median prediction
ref_idx = pred_series.iloc[
    (pred_series - pred_series.median()).abs().argsort()[:1]
].index[0]

base_row = X_test.loc[[ref_idx]]
base_pred = predict_single(model, base_row)

print(f'Reference observation index: {ref_idx}')
print(f'Base prediction (% Silica Concentrate): {base_pred:.4f}')
print(f'Actual value: {y_test.loc[ref_idx]:.4f}')

# Show key operational feature values for the reference observation
ops = CFG['feature_groups']['feed'] + CFG['feature_groups']['flow'] + CFG['feature_groups']['pulp']
available_ops = [c for c in ops if c in base_row.columns]
print('\nKey operational features (reference):')
print(base_row[available_ops].T.rename(columns={ref_idx: 'Value'}).round(4))

---
## 3. Define What-If Scenarios

Each scenario modifies one or more operational variables relative to the reference.
The modifications are inspired by the SHAP analysis from notebook 03:
the variables with the highest SHAP values are the most impactful levers.

| Scenario | Change | Rationale |
|---|---|---|
| A | Amina Flow +5% | Increased collector → expect lower silica |
| B | Amina Flow −5% | Reduced collector → expect higher silica |
| C | Starch Flow −5% | Reduced depressant → could increase silica entrainment |
| D | Starch Flow +5% | More depressant → may improve iron recovery but risk over-depression |
| E | Ore Pulp pH +0.5 | Alkalinity increase → affects reagent efficiency |
| F | Ore Pulp pH −0.5 | Acidity increase → may reduce collector effectiveness |
| G | % Silica Feed −10% | Better ore quality → expect lower concentrate silica |
| H | Combined: Amina +5% + Starch −5% | Combined reagent adjustment |

In [ ]:
def make_scenario(name: str, description: str, row: pd.DataFrame) -> dict:
    return {'name': name, 'description': description, 'row': row}

scenarios = []

# Scenario A: Amina Flow +5%
if 'Amina Flow' in base_row.columns:
    scenarios.append(make_scenario(
        'A — Amina Flow +5%',
        '+5% Amina Flow (increased collector dose)',
        apply_pct_change(base_row, 'Amina Flow', +5.0),
    ))

# Scenario B: Amina Flow −5%
if 'Amina Flow' in base_row.columns:
    scenarios.append(make_scenario(
        'B — Amina Flow -5%',
        '-5% Amina Flow (reduced collector dose)',
        apply_pct_change(base_row, 'Amina Flow', -5.0),
    ))

# Scenario C: Starch Flow −5%
if 'Starch Flow' in base_row.columns:
    scenarios.append(make_scenario(
        'C — Starch Flow -5%',
        '-5% Starch Flow (reduced depressant dose)',
        apply_pct_change(base_row, 'Starch Flow', -5.0),
    ))

# Scenario D: Starch Flow +5%
if 'Starch Flow' in base_row.columns:
    scenarios.append(make_scenario(
        'D — Starch Flow +5%',
        '+5% Starch Flow (increased depressant dose)',
        apply_pct_change(base_row, 'Starch Flow', +5.0),
    ))

# Scenario E: Ore Pulp pH +0.5
if 'Ore Pulp pH' in base_row.columns:
    ph_row = base_row.copy()
    ph_row['Ore Pulp pH'] = float(ph_row['Ore Pulp pH'].iloc[0]) + 0.5
    scenarios.append(make_scenario(
        'E — Ore Pulp pH +0.5',
        'pH increased by 0.5 units (more alkaline)',
        ph_row,
    ))

# Scenario F: Ore Pulp pH −0.5
if 'Ore Pulp pH' in base_row.columns:
    ph_row2 = base_row.copy()
    ph_row2['Ore Pulp pH'] = float(ph_row2['Ore Pulp pH'].iloc[0]) - 0.5
    scenarios.append(make_scenario(
        'F — Ore Pulp pH -0.5',
        'pH decreased by 0.5 units (more acidic)',
        ph_row2,
    ))

# Scenario G: % Silica Feed −10%
if '% Silica Feed' in base_row.columns:
    scenarios.append(make_scenario(
        'G — % Silica Feed -10%',
        '-10% Silica Feed (better ore quality)',
        apply_pct_change(base_row, '% Silica Feed', -10.0),
    ))

# Scenario H: Combined Amina +5% + Starch −5%
if 'Amina Flow' in base_row.columns and 'Starch Flow' in base_row.columns:
    combined_row = apply_pct_change(base_row, 'Amina Flow', +5.0)
    combined_row = apply_pct_change(combined_row, 'Starch Flow', -5.0)
    scenarios.append(make_scenario(
        'H — Amina +5% & Starch -5%',
        'Combined: +5% Amina Flow and -5% Starch Flow',
        combined_row,
    ))

print(f'Defined {len(scenarios)} scenarios.')

---
## 4. Run Predictions for All Scenarios

In [ ]:
results_table = compare_scenarios(model, base_row, scenarios)
print('\n=== Scenario Comparison ===')
results_table

In [ ]:
# Save scenario results
save_scenario_table(results_table, CFG, filename='scenario_comparison.csv')

---
## 5. Visualisation

In [ ]:
# ── 5a. Scenario comparison bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

colours = ['steelblue' if d >= 0 else 'coral'
           for d in results_table['Delta_vs_Base']]

bars = ax.bar(
    results_table['Scenario'],
    results_table['Predicted_Silica'],
    color=colours, edgecolor='white', alpha=0.85
)

ax.axhline(base_pred, color='navy', lw=1.5, linestyle='--',
           label=f'Base: {base_pred:.3f}')

for bar, val in zip(bars, results_table['Predicted_Silica']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.3f}',
        ha='center', va='bottom', fontsize=8
    )

ax.set_title('Predicted % Silica Concentrate by Scenario', fontsize=13)
ax.set_ylabel('% Silica Concentrate (predicted)')
ax.set_xlabel('Scenario')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'scenario_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5b. Delta vs Base ─────────────────────────────────────────────────────
delta_df = results_table[results_table['Scenario'] != 'Base'].copy()

fig, ax = plt.subplots(figsize=(10, 5))
colours_delta = ['coral' if d > 0 else 'steelblue'
                 for d in delta_df['Delta_vs_Base']]
ax.barh(delta_df['Scenario'], delta_df['Delta_vs_Base'],
        color=colours_delta, edgecolor='white', alpha=0.85)
ax.axvline(0, color='navy', lw=1.2)
ax.set_title('Change in Predicted % Silica vs Base Scenario', fontsize=12)
ax.set_xlabel('Δ % Silica Concentrate (predicted)')

red_patch = mpatches.Patch(color='coral', label='Higher silica (worse quality)')
blue_patch = mpatches.Patch(color='steelblue', label='Lower silica (better quality)')
ax.legend(handles=[red_patch, blue_patch])

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'scenario_delta.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Sensitivity Sweep — Amina Flow

We sweep Amina Flow from −20% to +20% to understand the model's sensitivity
profile for this variable while holding all others constant.

In [ ]:
if 'Amina Flow' in base_row.columns:
    sweep_amina = sensitivity_sweep(
        model, base_row, 'Amina Flow',
        pct_range=np.linspace(-20, 20, 41)
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sweep_amina['pct_change'], sweep_amina['predicted_silica'],
            color='steelblue', lw=2)
    ax.axvline(0, color='navy', lw=1.2, linestyle='--', label='Reference')
    ax.set_title('Sensitivity of % Silica Predict. to Amina Flow (ceteris paribus)', fontsize=11)
    ax.set_xlabel('% Change in Amina Flow')
    ax.set_ylabel('Predicted % Silica Concentrate')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / 'sensitivity_amina_flow.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(sweep_amina)

In [ ]:
# Sensitivity sweep for Ore Pulp pH
if 'Ore Pulp pH' in base_row.columns:
    sweep_ph = sensitivity_sweep(
        model, base_row, 'Ore Pulp pH',
        pct_range=np.linspace(-10, 10, 41)
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sweep_ph['pct_change'], sweep_ph['predicted_silica'],
            color='coral', lw=2)
    ax.axvline(0, color='navy', lw=1.2, linestyle='--', label='Reference')
    ax.set_title('Sensitivity of % Silica Predict. to Ore Pulp pH (ceteris paribus)', fontsize=11)
    ax.set_xlabel('% Change in Ore Pulp pH')
    ax.set_ylabel('Predicted % Silica Concentrate')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / 'sensitivity_ore_pulp_ph.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 7. Trade-offs, Constraints, and Risks

### What the scenarios suggest
The model predicts that:
- **Increasing Amina Flow** tends to reduce predicted % Silica Concentrate — potentially
  improving concentrate quality. However, excess amina increases reagent costs and can
  cause froth stability problems.
- **Reducing Starch Flow** may increase silica entrainment — worse quality.
- **Combined reagent adjustment** (Amina +5%, Starch −5%) shows interactions that a
  univariate model cannot fully capture.

### Key constraints not captured by the model
| Constraint | Description |
|---|---|
| Reagent dosage limits | Reagents cannot be increased indefinitely; physical constraints and cost limits apply |
| Process non-linearity | The flotation process has complex non-linear dynamics; model was trained on historical ranges |
| Distribution shift | If we push variables outside the training data range, predictions become unreliable |
| Interactions | Changing one variable affects others (pH changes affect reagent effectiveness); the model partially captures this through learned associations but not mechanistically |
| Recovery trade-off | Reducing silica in concentrate often reduces iron recovery — a critical metallurgical trade-off |

### Risks
1. **Extrapolation**: Scenarios that push variables outside the observed historical range
   (e.g., pH ±10%) are outside the model's training distribution. Predictions in these
   regions have high uncertainty.
2. **Ceteris paribus assumption**: Sensitivity sweeps change one variable while fixing all
   others. In reality, operational variables are correlated and interactive.
3. **Lag features**: The model includes lagged values of variables. Changing the current
   value does not change the lags — the scenario is therefore slightly inconsistent with
   a real sustained change in operations.
4. **Model uncertainty**: We report point predictions; a production system should also
   report prediction intervals (e.g., using conformal prediction or quantile regression).

### Recommended next steps for operationalisation
1. Validate model predictions in a controlled A/B test with process engineers.
2. Add prediction uncertainty quantification.
3. Integrate with real-time process historian (PI, OSIsoft) for online inference.
4. Implement drift monitoring to detect when the model needs retraining.

---
## Summary — Level 4 Checklist

| Item | Status |
|---|---|
| Reference observation selected from test set | ✅ |
| Multiple what-if scenarios defined (A–H) | ✅ |
| Scenarios cover: Amina, Starch, pH, Silica Feed, combined | ✅ |
| Comparison table with delta vs base | ✅ |
| Scenario bar chart and delta chart | ✅ |
| Sensitivity sweep (Amina, pH) | ✅ |
| Results saved to `reports/metrics/` | ✅ |
| Figures saved to `reports/figures/` | ✅ |
| Trade-offs and risks discussed | ✅ |
| Explicit disclaimer: simulation ≠ causal recommendation | ✅ |